In [1]:
# Cell 1: imports, plotting, and path setup
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from sklearn.linear_model import LinearRegression
import numpy as np

PROJECT_ROOT = Path("..").resolve()

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.autolayout"] = True

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())
print("Torch version:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Project root: <project root>
Working directory: <project root>/notebooks
Torch version: 2.11.0+cu130
Using device: cuda


In [2]:
# Cell 2: import project code
from src.data_loading import load_daily_clean_nuuksio
from src.models import MLPRegressor
from src.training import (
    train_baseline,
    DEFAULT_TRAIN_CONFIG,
    eval_loader_reg,
)

In [3]:
# Cell 3: load the clean daily Nuuksio dataset
CLEAN_PATH = Path("../data/processed/nuuksio_daily_clean.csv")

df = load_daily_clean_nuuksio(CLEAN_PATH)
print("Loaded clean daily data:", df.shape)
df.head()

Loaded clean daily data: (3653, 10)


,datetime,date,station,tmax,tmin,tavg,prcp,dayofyear,dayofyear_sin,dayofyear_cos
0,2016-01-01 02:00:00,2016-01-01,Espoo Nuuksio,-4.3,-6.1,-5.6,0.2,1,0.000000,1.000000
1,2016-01-02 02:00:00,2016-01-02,Espoo Nuuksio,-6.0,-11.1,-9.5,0.4,2,0.017202,0.999852
2,2016-01-03 02:00:00,2016-01-03,Espoo Nuuksio,-8.6,-13.1,-10.3,4.5,3,0.034398,0.999408
3,2016-01-04 02:00:00,2016-01-04,Espoo Nuuksio,-7.5,-15.3,-13.4,0.0,4,0.051584,0.998669
4,2016-01-05 02:00:00,2016-01-05,Espoo Nuuksio,-15.2,-21.6,-19.6,0.1,5,0.068755,0.997634


In [4]:
# Cell 4: sanity checks – sorted by time, no obvious duplicates
# Purpose:
#   - Confirm the dataset covers the expected time range
#   - Confirm there are no missing values in the core numeric columns
#   - Ensure sorting by datetime (important because we do a temporal split later
print("Datetime min/max:", df["datetime"].min(), "→", df["datetime"].max())
print("Number of unique dates:", df["date"].nunique())
print("Number of rows:", len(df))

if not df["datetime"].is_monotonic_increasing:
    print("Sorting dataframe by datetime...")
    df = df.sort_values("datetime").reset_index(drop=True)

numeric_cols = ["tmax", "tmin", "tavg", "prcp"]
print("Missing values per numeric column:")
print(df[numeric_cols].isna().sum())

Datetime min/max: 2016-01-01 02:00:00 → 2025-12-31 02:00:00
Number of unique dates: 3653
Number of rows: 3653
Missing values per numeric column:
tmax    0
tmin    0
tavg    0
prcp    0
dtype: int64


In [5]:
# Cell 5: build next-day modelling dataset
from src.training import make_standard_experiment_data

config = DEFAULT_TRAIN_CONFIG
data   = make_standard_experiment_data(PROJECT_ROOT, config)

train_loader = data.train_loader
val_loader   = data.val_loader
test_loader  = data.test_loader
feature_cols = data.feature_cols
train_df     = data.train_df
val_df       = data.val_df
test_df      = data.test_df

print("Train/val/test sizes:", len(train_df), len(val_df), len(test_df))
print("Feature dimension:", len(feature_cols))

Train/val/test sizes: 2556 547 549
Feature dimension: 7


In [6]:
# Cell 5A: verify temporal split boundaries and chronology

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "n_rows": len(train_df),
            "start_datetime": pd.to_datetime(train_df["datetime"]).min(),
            "end_datetime": pd.to_datetime(train_df["datetime"]).max(),
        },
        {
            "split": "val",
            "n_rows": len(val_df),
            "start_datetime": pd.to_datetime(val_df["datetime"]).min(),
            "end_datetime": pd.to_datetime(val_df["datetime"]).max(),
        },
        {
            "split": "test",
            "n_rows": len(test_df),
            "start_datetime": pd.to_datetime(test_df["datetime"]).min(),
            "end_datetime": pd.to_datetime(test_df["datetime"]).max(),
        },
    ]
)

print("Temporal split summary:")
display(split_summary)

train_end = pd.to_datetime(train_df["datetime"]).max()
val_start = pd.to_datetime(val_df["datetime"]).min()
val_end = pd.to_datetime(val_df["datetime"]).max()
test_start = pd.to_datetime(test_df["datetime"]).min()

print("\nChronology checks:")
print("train_end < val_start :", train_end < val_start)
print("val_end   < test_start:", val_end < test_start)
print("full dataset sorted   :", data.train_df['datetime'].is_monotonic_increasing 
      and data.val_df['datetime'].is_monotonic_increasing
      and data.test_df['datetime'].is_monotonic_increasing)

Temporal split summary:


,split,n_rows,start_datetime,end_datetime
0,train,2556,2016-01-01 02:00:00,2022-12-30 02:00:00
1,val,547,2022-12-31 02:00:00,2024-06-29 03:00:00
2,test,549,2024-06-30 03:00:00,2025-12-30 02:00:00



Chronology checks:
train_end < val_start : True
val_end   < test_start: True
full dataset sorted   : True


In [7]:
# Cell 5B: compact split table for thesis use

split_table = split_summary.copy()
split_table["start_datetime"] = pd.to_datetime(split_table["start_datetime"]).dt.strftime("%Y-%m-%d")
split_table["end_datetime"] = pd.to_datetime(split_table["end_datetime"]).dt.strftime("%Y-%m-%d")

split_table

,split,n_rows,start_datetime,end_datetime
0,train,2556,2016-01-01,2022-12-30
1,val,547,2022-12-31,2024-06-29
2,test,549,2024-06-30,2025-12-30


### Why temporal split?

The task is next-day forecasting, so the model should be trained on earlier observations and evaluated on later observations. A temporal split matches the intended use case more closely than a random split and avoids mixing nearby time points across train, validation, and test sets.

In [8]:
# Cell 5C - helper metrics for numpy / sklearn baselines
def eval_reg_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64).ravel()
    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()

    mse = float(np.mean((y_pred - y_true) ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(y_pred - y_true)))

    return {"mse": mse, "rmse": rmse, "mae": mae}

In [9]:
# Cell 5D - persistence baseline: predict next-day tmax using same-day tmax
# y_hat_{t+1} = tmax_t

yhat_train_persist = train_df["tmax"].to_numpy(dtype=float)
yhat_val_persist   = val_df["tmax"].to_numpy(dtype=float)
yhat_test_persist  = test_df["tmax"].to_numpy(dtype=float)

persist_train_metrics = eval_reg_np(data.y_train, yhat_train_persist)
persist_val_metrics   = eval_reg_np(data.y_val,   yhat_val_persist)
persist_test_metrics  = eval_reg_np(data.y_test,  yhat_test_persist)

print("Persistence baseline metrics")
print("Train:", persist_train_metrics)
print("Val:  ", persist_val_metrics)
print("Test: ", persist_test_metrics)

persistence_metrics_table = pd.DataFrame(
    [
        {"split": "train", **persist_train_metrics},
        {"split": "val",   **persist_val_metrics},
        {"split": "test",  **persist_test_metrics},
    ]
).round(4)

persistence_metrics_table

Persistence baseline metrics
Train: {'mse': 7.204143192488263, 'rmse': 2.6840535003029027, 'mae': 2.003716744913928}
Val:   {'mse': 7.371005484460696, 'rmse': 2.714959573264526, 'mae': 2.0307129798903105}
Test:  {'mse': 6.995646630236794, 'rmse': 2.6449284735578003, 'mae': 1.9972677595628416}


,split,mse,rmse,mae
0,train,7.2041,2.6841,2.0037
1,val,7.3710,2.7150,2.0307
2,test,6.9956,2.6449,1.9973


In [10]:
# Cell 5E - linear regression baseline on the same standardized feature matrix
linreg_model = LinearRegression()
linreg_model.fit(data.X_train, data.y_train)

yhat_train_linreg = linreg_model.predict(data.X_train)
yhat_val_linreg   = linreg_model.predict(data.X_val)
yhat_test_linreg  = linreg_model.predict(data.X_test)

linreg_train_metrics = eval_reg_np(data.y_train, yhat_train_linreg)
linreg_val_metrics   = eval_reg_np(data.y_val,   yhat_val_linreg)
linreg_test_metrics  = eval_reg_np(data.y_test,  yhat_test_linreg)

print("Linear regression baseline metrics")
print("Train:", linreg_train_metrics)
print("Val:  ", linreg_val_metrics)
print("Test: ", linreg_test_metrics)

linreg_metrics_table = pd.DataFrame(
    [
        {"split": "train", **linreg_train_metrics},
        {"split": "val",   **linreg_val_metrics},
        {"split": "test",  **linreg_test_metrics},
    ]
).round(4)

linreg_metrics_table

Linear regression baseline metrics
Train: {'mse': 5.715033345050649, 'rmse': 2.3906135917480786, 'mae': 1.8318882598449298}
Val:   {'mse': 6.023682775166035, 'rmse': 2.4543192080831773, 'mae': 1.9391915437776404}
Test:  {'mse': 5.357829230693768, 'rmse': 2.314698518315888, 'mae': 1.8133933661316401}


,split,mse,rmse,mae
0,train,5.7150,2.3906,1.8319
1,val,6.0237,2.4543,1.9392
2,test,5.3578,2.3147,1.8134


In [11]:
# Cell 5F - linear regression coefficients (standardized-feature space)
linreg_coef_table = pd.DataFrame(
    {
        "feature": feature_cols,
        "coefficient": linreg_model.coef_.astype(float),
        "abs_coefficient": np.abs(linreg_model.coef_.astype(float)),
    }
).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

linreg_intercept = float(linreg_model.intercept_)

print(f"Linear regression intercept: {linreg_intercept:.4f}")
linreg_coef_table.round(4)

Linear regression intercept: 9.8592


,feature,coefficient,abs_coefficient
0,tavg,9.2476,9.2476
1,tmin,-2.8433,2.8433
2,dayofyear_cos,-2.4567,2.4567
3,tmax,0.8228,0.8228
4,prcp,-0.1891,0.1891
5,dayofyear_sin,-0.1160,0.1160
6,rand_noise,0.0240,0.0240


In [12]:
# Cell 6: create baseline MLP model and training config, then train
# Purpose:
#   - Train a plain MLPRegressor to minimise next-day prediction MSE
#   - This is the reference model for later interpretability-regularised variants

# Architecture choice: (64, 64) ReLU MLP
# (dropout=0.0 here; weight decay + early stopping are used for regularisation)
input_dim = len(feature_cols)
model = MLPRegressor(
    input_dim=input_dim,
    hidden_dims=(64, 64),
    dropout=0.0,
)

config = DEFAULT_TRAIN_CONFIG

# train_baseline:
#   - optimizer: AdamW (see src/training.py)
#   - loss: MSE
#   - early stopping on validation loss (patience set in DEFAULT_TRAIN_CONFIG)

# How to read the baseline training log:
#
# Example:
# Epoch 010 | train_loss=6.2170 | val_loss=6.9680
#
# Meaning:
# - train_loss: average training MSE during that epoch
# - val_loss:   average validation MSE during that epoch
#
# These values are stored in `history` and can be shown either as a plot
# or as a table.

model, history = train_baseline(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
)

Epoch 001 | train_loss=178.9841 | val_loss=160.5828
Epoch 010 | train_loss=6.3977 | val_loss=7.1469
Epoch 020 | train_loss=5.7435 | val_loss=6.2852
Epoch 030 | train_loss=5.5587 | val_loss=6.0155
Epoch 040 | train_loss=5.4628 | val_loss=6.0284
Early stopping at epoch 046 | best_val_loss=5.8916


In [13]:
# Cell F - benchmark comparison table
baseline_mlp_train_metrics = eval_loader_reg(model, train_loader, device=device)
baseline_mlp_val_metrics   = eval_loader_reg(model, val_loader,   device=device)
baseline_mlp_test_metrics  = eval_loader_reg(model, test_loader,  device=device)

benchmark_comparison_table = pd.DataFrame(
    [
        {
            "model": "Persistence (tmax_t -> tmax_{t+1})",
            "train_rmse": persist_train_metrics["rmse"],
            "val_rmse": persist_val_metrics["rmse"],
            "test_rmse": persist_test_metrics["rmse"],
            "test_mae": persist_test_metrics["mae"],
        },
        {
            "model": "Linear regression",
            "train_rmse": linreg_train_metrics["rmse"],
            "val_rmse": linreg_val_metrics["rmse"],
            "test_rmse": linreg_test_metrics["rmse"],
            "test_mae": linreg_test_metrics["mae"],
        },
        {
            "model": "Baseline MLP (ERM)",
            "train_rmse": baseline_mlp_train_metrics["rmse"],
            "val_rmse": baseline_mlp_val_metrics["rmse"],
            "test_rmse": baseline_mlp_test_metrics["rmse"],
            "test_mae": baseline_mlp_test_metrics["mae"],
        },
    ]
).round(4)

benchmark_comparison_table

,model,train_rmse,val_rmse,test_rmse,test_mae
0,Persistence (tmax_t -> tmax_{t+1}),2.6841,2.7150,2.6449,1.9973
1,Linear regression,2.3906,2.4543,2.3147,1.8134
2,Baseline MLP (ERM),2.3327,2.4273,2.3038,1.8117


In [14]:
# Cell 7: Helper cell: safe scalar coercion for pandas / numpy values
import math
from typing import Any

import numpy as np
import pandas as pd


def to_py_float(value: Any, *, name: str) -> float:
    """
    Convert a pandas/numpy scalar to a validated Python float.

    Raises
    ------
    ValueError
        If the value is missing or not finite.
    """
    arr = pd.to_numeric(pd.Series([value]), errors="raise").to_numpy(dtype=np.float64)
    out = float(arr.item())

    if not math.isfinite(out):
        raise ValueError(f"{name} must be finite, got {out!r}")

    return out


def to_py_int(value: Any, *, name: str) -> int:
    """
    Convert a pandas/numpy scalar to a validated Python int.

    Raises
    ------
    ValueError
        If the value is missing, not finite, or not integer-valued.
    """
    out = to_py_float(value, name=name)

    if not out.is_integer():
        raise ValueError(f"{name} must be integer-valued, got {out!r}")

    return int(out)

In [15]:
# Cell 8: Replacement for the shortened baseline training-history summary cell
history_df = pd.DataFrame(history)

best_idx = history_df["val_loss"].idxmin()

best_epoch = to_py_int(history_df.at[best_idx, "epoch"], name="best_epoch")
final_epoch = to_py_int(history_df["epoch"].max(), name="final_epoch")

best_val_loss = to_py_float(history_df.at[best_idx, "val_loss"], name="best_val_loss")
train_loss_at_best_val = to_py_float(
    history_df.at[best_idx, "train_loss"],
    name="train_loss_at_best_val",
)
final_train_loss = to_py_float(history_df.iloc[-1]["train_loss"], name="final_train_loss")
final_val_loss = to_py_float(history_df.iloc[-1]["val_loss"], name="final_val_loss")

milestone_epochs = sorted(set([0, 1, 2, 5, 10, 20, 30, best_epoch, final_epoch]))

baseline_training_summary = pd.DataFrame(
    [
        {
            "best_val_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "train_loss_at_best_val": train_loss_at_best_val,
            "final_epoch": final_epoch,
            "final_train_loss": final_train_loss,
            "final_val_loss": final_val_loss,
            "n_epochs_trained": len(history_df),
        }
    ]
).round(4)

baseline_history_milestones = (
    history_df.loc[history_df["epoch"].isin(milestone_epochs), ["epoch", "train_loss", "val_loss"]]
    .sort_values("epoch")
    .reset_index(drop=True)
    .round(4)
)

print("Baseline training summary")
display(baseline_training_summary)

print("Baseline training history at milestone epochs")
display(baseline_history_milestones)

Baseline training summary


,best_val_epoch,best_val_loss,train_loss_at_best_val,final_epoch,final_train_loss,final_val_loss,n_epochs_trained
0,35,5.8916,5.4888,45,5.4314,6.0439,46


Baseline training history at milestone epochs


,epoch,train_loss,val_loss
0,0,178.9841,160.5828
1,1,97.1007,39.6887
2,2,12.7884,9.6141
3,5,7.0536,8.0844
4,10,6.2884,6.9567
5,20,5.7060,6.3042
6,30,5.5557,5.9635
7,35,5.4888,5.8916
8,45,5.4314,6.0439


## Metrics used in this notebook

- **MSE**: mean squared error
- **RMSE**: square root of MSE
- **MAE**: mean absolute error

These are reported on the train, validation, and test splits.

The baseline model has no interpretability regularization; it is the ERM reference.

In [16]:
# Cell 9 - Evaluate baseline performance and save results
from src.results import save_experiment_results

train_metrics = eval_loader_reg(model, train_loader, device=device)
val_metrics   = eval_loader_reg(model, val_loader,   device=device)
test_metrics  = eval_loader_reg(model, test_loader,  device=device)

print("Train metrics:", train_metrics)
print("Val metrics:  ", val_metrics)
print("Test metrics: ", test_metrics)

config_dict = {
    "train_config": {
        "lr": config.lr,
        "batch_size": config.batch_size,
        "eval_batch_size": config.eval_batch_size,
        "num_epochs": config.num_epochs,
    },
    "feature_cols": feature_cols,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "device": device,
    "early_stopping_monitor": "val_loss",
}

results_dir = save_experiment_results(
    results_dir=PROJECT_ROOT / "results" / "baseline",
    model=model,
    config_dict=config_dict,
    metrics={"train": train_metrics, "val": val_metrics, "test": test_metrics},
    history=history,
)
results_dir

Train metrics: {'mse': 5.441389331020964, 'rmse': 2.332678574304862, 'mae': 1.775076184882338}
Val metrics:   {'mse': 5.8915807173585755, 'rmse': 2.4272578596759296, 'mae': 1.883178833279575}
Test metrics:  {'mse': 5.307536243269416, 'rmse': 2.3038090726597584, 'mae': 1.811691300275113}


PosixPath('<project root>/results/baseline')

In [17]:
# Cell 11 - Numeric table for final baseline metrics
baseline_metrics_table = pd.DataFrame(
    [
        {"split": "train", **train_metrics},
        {"split": "val",   **val_metrics},
        {"split": "test",  **test_metrics},
    ]
).round(4)

baseline_metrics_table

,split,mse,rmse,mae
0,train,5.4414,2.3327,1.7751
1,val,5.8916,2.4273,1.8832
2,test,5.3075,2.3038,1.8117
